# Baseline: طبقه‌بندی ژانر موسیقی با MFCC و SVM

این نوت‌بوک نسخه صوتیِ آزمایش کلاسی است: ورودی‌ها را بارگذاری می‌کند، ویژگی استخراج می‌کند، یک روش پایه را با SVM مقایسه می‌کند و نتیجه را با معیارهای مناسب گزارش می‌دهد. پیش از اجرا، داده‌ها را طبق data/README.md قرار دهید.

## 1. کتابخانه‌ها و تنظیمات

برای تکرارپذیری، نرخ نمونه‌برداری و seed ثابت هستند. این baseline هر فایل را به mono و حداکثر ۳۰ ثانیه تبدیل می‌کند.

In [ ]:
from pathlib import Path
import time

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

RANDOM_STATE = 42
SAMPLE_RATE = 22050
N_MFCC = 20
DURATION_SECONDS = 30
DATA_DIR = Path("data")
AUDIO_DIR = DATA_DIR / "audio"
METADATA_PATH = DATA_DIR / "metadata.csv"

## 2. بارگذاری و بررسی فراداده

metadata.csv باید ستون‌های filename و genre داشته باشد. برای تقسیم آموزش و آزمون، هر ژانر باید دست‌کم پنج فایل داشته باشد.

In [ ]:
if not METADATA_PATH.exists():
    raise FileNotFoundError(
        "data/metadata.csv پیدا نشد. قالب داده را در data/README.md ببینید."
    )

metadata_df = pd.read_csv(METADATA_PATH)
required_columns = {"filename", "genre"}
missing_columns = required_columns - set(metadata_df.columns)
if missing_columns:
    raise ValueError(f"ستون‌های لازم وجود ندارند: {sorted(missing_columns)}")

metadata_df = metadata_df.dropna(subset=["filename", "genre"]).copy()
metadata_df["filename"] = metadata_df["filename"].astype(str)
metadata_df["genre"] = metadata_df["genre"].astype(str)
metadata_df["path"] = metadata_df["filename"].map(lambda name: AUDIO_DIR / name)
missing_files = metadata_df.loc[~metadata_df["path"].map(Path.exists), "filename"]
if not missing_files.empty:
    raise FileNotFoundError(f"فایل‌های صوتی پیدا نشدند: {missing_files.head(10).tolist()}")

class_counts = metadata_df["genre"].value_counts().sort_index()
display(class_counts.rename_axis("genre").to_frame("number_of_clips"))
if class_counts.empty or class_counts.min() < 5:
    raise ValueError("برای هر ژانر دست‌کم پنج فایل صوتی فراهم کنید.")

## 3. دیدن موج صوتی و MFCC

نمودارها یک نمونه از داده واقعی پروژه را نشان می‌دهند و کمک می‌کنند بارگذاری و استخراج ویژگی بررسی شود.

In [ ]:
example_path = metadata_df.iloc[0]["path"]
waveform, sr = librosa.load(
    example_path, sr=SAMPLE_RATE, mono=True, duration=DURATION_SECONDS
)
print("Example file:", example_path.name)
print("Genre label:", metadata_df.iloc[0]["genre"])
print("Sample rate:", sr, "Hz")

fig, axes = plt.subplots(2, 1, figsize=(12, 6), constrained_layout=True)
librosa.display.waveshow(waveform, sr=sr, ax=axes[0])
axes[0].set_title("Audio waveform")
mfcc_example = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=N_MFCC)
img = librosa.display.specshow(mfcc_example, x_axis="time", sr=sr, ax=axes[1])
axes[1].set_title("MFCC features")
fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
plt.show()

## 4. استخراج ویژگی از همه فایل‌ها

برای هر فایل، میانگین و انحراف معیار زمانی هر یک از ۲۰ ضریب MFCC به‌عنوان ویژگی استفاده می‌شود.

In [ ]:
def extract_mfcc_summary(path):
    signal, sr = librosa.load(
        path, sr=SAMPLE_RATE, mono=True, duration=DURATION_SECONDS
    )
    if signal.size == 0:
        raise ValueError("فایل صوتی خالی است.")
    mfcc = librosa.feature.mfcc(y=signal, sr=sr, n_mfcc=N_MFCC)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])

feature_rows = []
labels = []
failed_files = []
for row in metadata_df.itertuples(index=False):
    try:
        feature_rows.append(extract_mfcc_summary(row.path))
        labels.append(row.genre)
    except Exception as error:
        failed_files.append((row.filename, str(error)))

if failed_files:
    print("فایل‌های ردشده:", failed_files[:10])
if not feature_rows:
    raise ValueError("هیچ ویژگی صوتی استخراج نشد. فایل‌ها و فراداده را بررسی کنید.")

X = np.vstack(feature_rows)
y = np.asarray(labels)
print("Feature matrix shape:", X.shape)
print("Number of labels:", y.size)

## 5. مقایسه baseline و SVM

هر دو روش از همان تقسیم آموزش/آزمون استفاده می‌کنند. اگر داده چند کلیپ از یک آهنگ منبع دارد، پیش از تحویل، تقسیم را بر اساس شناسه آهنگ انجام دهید تا کلیپ‌های یک آهنگ دو طرف تقسیم نشوند.

In [ ]:
indices = np.arange(len(y))
train_idx, test_idx = train_test_split(
    indices, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

svm_model = make_pipeline(
    StandardScaler(),
    SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE),
)
start = time.perf_counter()
svm_model.fit(X_train, y_train)
svm_fit_seconds = time.perf_counter() - start
start = time.perf_counter()
svm_predictions = svm_model.predict(X_test)
svm_predict_seconds = time.perf_counter() - start

results = pd.DataFrame(
    [
        {"model": "Most frequent baseline", "accuracy": accuracy_score(y_test, baseline_predictions),
         "macro_f1": f1_score(y_test, baseline_predictions, average="macro", zero_division=0)},
        {"model": "MFCC + SVM", "accuracy": accuracy_score(y_test, svm_predictions),
         "macro_f1": f1_score(y_test, svm_predictions, average="macro", zero_division=0),
         "fit_seconds": svm_fit_seconds, "predict_seconds": svm_predict_seconds},
    ]
)
display(results.round(3))
print(classification_report(y_test, svm_predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, svm_predictions, xticks_rotation=45, cmap="Purples"
)
plt.title("MFCC + SVM: confusion matrix")
plt.tight_layout()
plt.show()

## نتیجه و محدودیت‌ها

نتیجه را با Macro-F1، Accuracy و ماتریس درهم‌ریختگی گزارش کنید. این baseline برای شروع است؛ کیفیت آن به تعداد و تنوع فایل‌ها، برچسب‌گذاری و جداسازی درست داده بستگی دارد. پیش از تحویل، منبع و مجوز داده، تعداد نمونه‌های هر ژانر و هر فایل ناموفق را در README ثبت کنید.